# Logo Images Embedding

## Imports

In [ ]:
!pip install datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.2 MB/s eta 0:00:00


In [ ]:
zip_dir = r"/content/drive/MyDrive/logo_recognition_similarity_search_project/logo_images_zip"
unzip_dir = "/content"
!ls {zip_dir}

ls: cannot access '/content/drive/MyDrive/logo_recognition_similarity_search_project/logo_images_zip': No such file or directory


In [ ]:
num = 8
directory_name = f"images{num}"
path = f"{zip_dir}/{directory_name}.zip"

In [ ]:
!unzip {path} -d {unzip_dir}
print(f"unzip {directory_name}.zip complete")

unzip:  cannot find or open /content/drive/MyDrive/logo_recognition_similarity_search_project/logo_images_zip/images8.zip, /content/drive/MyDrive/logo_recognition_similarity_search_project/logo_images_zip/images8.zip.zip or /content/drive/MyDrive/logo_recognition_similarity_search_project/logo_images_zip/images8.zip.ZIP.
unzip images8.zip complete


In [ ]:
import os
import torchvision
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import torch
import datasets
import warnings
warnings.filterwarnings('ignore')



logo_dir = r"/content"
captions_path = r"/content/drive/MyDrive/Data/logo_image_caption.tsv"
output_dir = f"/content/drive/MyDrive/logo_recognition_similarity_search_project/output/logo_images_embeddings"
os.makedirs(output_dir, exist_ok=True)

images_dir = f"{logo_dir}/{directory_name}"
print("Current Images Directory:", images_dir)
!ls {images_dir}

Current Images Directory: /content/images8
ls: cannot access '/content/images8': No such file or directory


## Dataset &  ResNet-50 Embedding

In [ ]:
# Define transformations to preprocess images for ResNet-50
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Define the LogoDataset class
class LogoDataset(Dataset):
    def __init__(self, root_dir, transform):
        """
        Custom PyTorch dataset for loading logo images.
        :param root_dir: The root directory containing all image subdirectories
        :param transform: Transformations to be applied to each image
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = self._get_all_image_paths()
        # Load ResNet-50 model
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = models.resnet50(pretrained=True)
        self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
        self.model = self.model.to(self.device)
        self.model.eval()
        self.image_embeddings = self._get_all_image_embeddings(self.image_paths)

    def _get_all_image_paths(self):
        """
        Traverse all subdirectories to find and list all logo images.
        :return: List of full paths to all images.
        """
        image_paths = []
        for subdir, _, files in os.walk(self.root_dir):
          for file in files:
              if file.lower().endswith(".png"):
                  image_paths.append(os.path.join(subdir, file))
        return image_paths

    def _get_all_image_embeddings(self,image_paths):
        """
        Extract embeddings for all images in the dataset.
        :return: List of embeddings for all images.
        """
        image_embeddings = []
        for image_path in tqdm(image_paths, desc="Extracting Embeddings"):
            image = Image.open(image_path).convert("RGB")
            image = self.transform(image)
            with torch.no_grad():
                image_embedding = self.model(image.unsqueeze(0).to(self.device)).squeeze().cpu().numpy()
            image_embeddings.append(image_embedding)
        return image_embeddings

    def __len__(self):
        """Returns the total number of images in the dataset."""
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Load an image by its index and apply necessary transformations.
        :param idx: Index of the image
        :return: A tuple containing the transformed image tensor and its file path
        """
        image_path = self.image_paths[idx]
        image_embedding = self.image_embeddings[idx]

        return image_embedding, image_path


In [ ]:
# Initialize dataset
dataset = LogoDataset(root_dir=images_dir, transform=transform)
ds = datasets.Dataset.from_dict({"path": dataset.image_paths,"embedding": dataset.image_embeddings})
ds.save_to_disk(f"{output_dir}/{directory_name}_logo_images_embeddings")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s]
Extracting Embeddings: 0it [00:00, ?it/s]


Saving the dataset (0/1 shards): 0 examples [00:00, ? examples/s]

In [ ]:
ds

In [ ]:
ds.features